# NB 01 — Data Acquisition

**Phase 1 — Tasks 1.3 / 1.4**

In this notebook:
- Downloads KFUPM-JRCAI/arabic-generated-abstracts from Hugging Face
- Builds a binary-labelled Spark DataFrame (human=0, AI=1)
- Runs initial schema and distribution EDA
- Saves raw checkpoint to GDrive

In [1]:
# ── Session bootstrap ─────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, importlib
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection')
SRC_DIR = PROJECT_ROOT / 'src'

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project root not found: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# src/__init__.py is committed to the repo, no runtime touch needed
print('PROJECT_ROOT:', PROJECT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection


In [ ]:
from src.utils import create_spark_session, checkpoint_exists, load_checkpoint, save_checkpoint, logger
from src.utils import show_class_distribution, sample_texts, DATA_RAW, add_src_to_spark
from src.data_preparation import download_dataset, build_labelled_dataframe
from pyspark.sql import functions as F

spark = create_spark_session('ArabicAIDetection_DataAcq')
add_src_to_spark(spark)


'/content/drive/MyDrive/MSBDA-801-Project/arabic_ai_detection/src_package.zip'

## Task 1.3 — Download dataset from Hugging Face

In [ ]:
raw_files = list(DATA_RAW.glob('*.parquet'))
if len(raw_files) < 3:
    print('Downloading dataset …')
    raw_dfs = download_dataset()
    print(f'Downloaded {sum(len(d) for d in raw_dfs.values()):,} rows across {len(raw_dfs)} splits.')
else:
    print(f'Raw data already present: {[f.name for f in raw_files]}')


README.md: 0.00B [00:00, ?B/s]

data/by_polishing-00000-of-00001.parquet:   0%|          | 0.00/8.49M [00:00<?, ?B/s]

data/from_title-00000-of-00001.parquet:   0%|          | 0.00/6.90M [00:00<?, ?B/s]

data/from_title_and_content-00000-of-000(…):   0%|          | 0.00/7.01M [00:00<?, ?B/s]

Generating by_polishing split:   0%|          | 0/2851 [00:00<?, ? examples/s]

Generating from_title split:   0%|          | 0/2963 [00:00<?, ? examples/s]

Generating from_title_and_content split:   0%|          | 0/2574 [00:00<?, ? examples/s]

Downloaded 8,388 rows across 3 splits.


## Build binary-labelled Spark DataFrame

In [ ]:
if checkpoint_exists('labelled_raw'):
    labelled_df = load_checkpoint(spark, 'labelled_raw')
    print('Loaded from checkpoint.')
else:
    labelled_df = build_labelled_dataframe(spark)
    save_checkpoint(labelled_df, 'labelled_raw')

labelled_df.cache()
labelled_df.printSchema()


root
 |-- text: string (nullable = true)
 |-- label: integer (nullable = false)
 |-- source_model: string (nullable = false)
 |-- generation_method: string (nullable = false)



## Task 1.4 — Initial EDA

In [ ]:
total = labelled_df.count()
print(f'Total rows: {total:,}')

print('\n── Class distribution ──')
show_class_distribution(labelled_df)

print('\n── Rows per source model ──')
labelled_df.groupBy('source_model').count().orderBy('count', ascending=False).show()

print('\n── Rows per generation method ──')
labelled_df.groupBy('generation_method').count().orderBy('count', ascending=False).show()


Total rows: 41,940

── Class distribution ──
+-----+-----+----+
|label|count| pct|
+-----+-----+----+
|    0| 8388|20.0|
|    1|33552|80.0|
+-----+-----+----+


── Rows per source model ──
+------------+-----+
|source_model|count|
+------------+-----+
|       human| 8388|
|       allam| 8388|
|        jais| 8388|
|       llama| 8388|
|      openai| 8388|
+------------+-----+


── Rows per generation method ──
+--------------------+-----+
|   generation_method|count|
+--------------------+-----+
|          from_title|14815|
|        by_polishing|14255|
|from_title_and_co...|12870|
+--------------------+-----+



In [ ]:
# Text length statistics by label
stats = (
    labelled_df
    .withColumn('text_len', F.length('text'))
    .groupBy('label')
    .agg(
        F.avg('text_len').alias('avg_len'),
        F.min('text_len').alias('min_len'),
        F.max('text_len').alias('max_len'),
        F.stddev('text_len').alias('std_len'),
    )
    .orderBy('label')
)
print('── Text length stats (chars) by label ──')
stats.show()


── Text length stats (chars) by label ──
+-----+-----------------+-------+-------+-----------------+
|label|          avg_len|min_len|max_len|          std_len|
+-----+-----------------+-------+-------+-----------------+
|    0|740.3141392465427|    411|   1891|226.5871823195146|
|    1|654.5988912732475|    170|  18459|261.4798001899716|
+-----+-----------------+-------+-------+-----------------+



In [ ]:
# Word count statistics by label
word_stats = (
    labelled_df
    .withColumn('word_count', F.size(F.split(F.col('text'), r'\s+')))
    .groupBy('label')
    .agg(
        F.avg('word_count').alias('avg_words'),
        F.min('word_count').alias('min_words'),
        F.max('word_count').alias('max_words'),
    )
    .orderBy('label')
)
print('── Word count stats by label ──')
word_stats.show()


── Word count stats by label ──
+-----+------------------+---------+---------+
|label|         avg_words|min_words|max_words|
+-----+------------------+---------+---------+
|    0|119.16452074391988|       75|      294|
|    1|101.49219122556032|       30|     2796|
+-----+------------------+---------+---------+



In [ ]:
# Null check
from pyspark.sql.functions import sum as Fsum, isnull
null_counts = labelled_df.select([
    Fsum(isnull(c).cast('int')).alias(c) for c in labelled_df.columns
])
print('── Null counts per column ──')
null_counts.show()


── Null counts per column ──
+----+-----+------------+-----------------+
|text|label|source_model|generation_method|
+----+-----+------------+-----------------+
|   0|    0|           0|                0|
+----+-----+------------+-----------------+



In [ ]:
# Sample texts
print('=== HUMAN sample ===')
sample_texts(labelled_df, n=2, label=0)
print('=== AI sample ===')
sample_texts(labelled_df, n=2, label=1)


=== HUMAN sample ===
─── Sample 1 [label=0, src=human] ───
كثيرا ما ارتبطت المصادر التاريخية في الأندلس خاصة منها كتب التراجم والفهرسات والبرامج وغيرها بدراسة حياة العلماء والرواة والقضاة والساسة ؛ وقد تطورت هذه المادة حتى ترك لنا المؤلفون الأندلسيون سلسلة متواصلة الحلقات من كتب التـراجم كالصلة لابن بشكوال ، وصلة الصلة لابن الزبير، والتكملة لكتاب الصلة لابن

─── Sample 2 [label=0, src=human] ───
يعد العامل الثقافي احد ابرز الاسباب التي يعزى لها سقوط الدولة الموحدية ، حتى أنه لايقل من حيث التأثير عن بقية العوامل خاصة السياسية و العسكرية ، فالتركيب الفكري لعقيدة ابن تومرت التي اعتبرت الركيزة الاساسية لقيام دولة الموحدين احتوى على تناقضات عقدية و فكرية و تشريعية فادحة ، جعلتها عرضة للانتقاد من

=== AI sample ===
─── Sample 1 [label=1, src=allam] ───
يتناول هذا البحث موضوع التعليم بين النساء الأندلسيات من خلال دراسة المصادر التاريخية المتعلقة بتراجم العلماء والرواة والقضاة والساسة. يركز البحث على إبراز دور المرأة العالمة ومساهمتها في الإنتاج الفكري والحضاري الأندلسي. من خلال تحليل كتب التر